# Module 3 — FaithEval dose-response + orthogonal-projection ablation (Llama-3.1-8B-Instruct)

**v2 §3 hypothesis, Llama parallel track.** Desperation steering increases hallucination on unanswerable-context questions; orthogonally projecting the desperation direction out of the residual stream decreases hallucination relative to baseline.

**Prerequisite:** M1 Llama (`outputs/m1_vectors/llama_L{layer}/desperation.npy`). M2 Llama is helpful for context (max usable α per the Llama capability gate) but M3 sweeps the M2 α grid regardless.

**Three arms:**
1. **Baseline** — Llama-3.1-8B-Instruct on FaithEval-unanswerable, no hook.
2. **Steered** — desperation hook at α ∈ {0.025, 0.05, 0.075, 0.1}. Expect hallucination to *increase* monotonically with α.
3. **Ablated** — orthogonal-projection hook (`h' = h − (h·v̂)v̂`). Expect hallucination to *decrease* relative to baseline. NO SAE dependency (the ablation is a hard projection of the M1 vector; SAE work is M4 only and stays on the Gemma side with the teammate).

**Pass criteria:**
- Monotonic dose-response in arm 2.
- Gap between arm 1 and arm 3 in the predicted direction (ablated ≤ baseline).

**Cost:** 2,492 FaithEval prompts × 6 runs ≈ 15k generations on 8B. ~6–8 hr A100 wall + ~$2 in Haiku classifier calls.

**Smoke-test:** set `LIMIT = 50` in Cell 2 for an end-to-end pipeline check.

**Scope note:** this notebook covers the *core* M3 protocol only. The Gemma M3 notebook has additional diagnostic cells (extended-α sweep, leakage scan, fabricate coherence audit, retention table) added in response to specific findings during the Gemma run. If the Llama run hits similar issues, port the relevant cells then — don't pre-build them.

## Cell 1 — env setup (Colab + SageMaker + local)

On SageMaker, expects `HF_TOKEN` and `ANTHROPIC_API_KEY` already in env, and that you opened this notebook from inside the cloned `Algoverse/` directory. The Anthropic key is required for the classifier judge on ambiguous outputs (~2% of generations).

In [ ]:
import os
import sys
import subprocess

# detect runtime
IS_COLAB = 'google.colab' in sys.modules
IS_SAGEMAKER = os.path.exists('/home/ec2-user/SageMaker') or 'SageMaker' in os.environ.get('PWD', '')
print(f'runtime: colab={IS_COLAB}, sagemaker={IS_SAGEMAKER}')

# secrets — Colab uses userdata; SageMaker/local expects them already in env.
# Two-token mode (optional): HF_MODEL_TOKEN authenticates the gated model download
# (e.g. a teammate's token who accepted the Llama license); HF_TOKEN authenticates
# Hub artifact-repo operations (your own token, with Write on the dataset).
# If HF_MODEL_TOKEN is unset, model_load falls back to HF_TOKEN for both.
if IS_COLAB:
	from google.colab import userdata
	os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
	try:
		os.environ['HF_MODEL_TOKEN'] = userdata.get('HF_MODEL_TOKEN')
	except Exception:
		pass  # single-token mode; model_load will use HF_TOKEN
	try:
		os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
	except Exception:
		print('warning: ANTHROPIC_API_KEY not in Colab userdata; M3 judge calls will fail')
else:
	assert 'HF_TOKEN' in os.environ, (
		'HF_TOKEN not set. Set it in the terminal (or at the top of this notebook):\n'
		'  export HF_TOKEN=hf_...   # token with Write on the artifact dataset repo\n'
		'Two-token mode (optional): also set HF_MODEL_TOKEN for the gated model download\n'
		'if a different account accepted the Llama license:\n'
		'  export HF_MODEL_TOKEN=hf_...\n'
		'Accept the model license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct'
	)
	if 'ANTHROPIC_API_KEY' not in os.environ:
		print('warning: ANTHROPIC_API_KEY not set; M3 judge calls (~ambiguous outputs) will fail. '
		      'export ANTHROPIC_API_KEY=sk-ant-... in terminal if you want them.')

# repo: Colab clones fresh each session; SageMaker/local expects you started in the repo
if IS_COLAB:
	subprocess.run('git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)', shell=True, check=True)
	os.chdir('Algoverse')
else:
	# Jupyter starts the kernel in the notebook's dir (notebooks/); walk up to repo root.
	if os.path.basename(os.getcwd()) == 'notebooks':
		os.chdir('..')
	assert os.path.isdir('src') and os.path.isfile('config.yaml'), (
		f'expected to be inside the Algoverse repo root, got cwd={os.getcwd()}. '
		'On SageMaker: `cd ~/SageMaker/Algoverse` and re-open this notebook from there.'
	)

# pip install is idempotent — fine to re-run on each session
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'cwd: {os.getcwd()}')


## Cell 1.4 — pre-flight: GPU, disk, dep check

In [ ]:
# pre-flight: GPU, disk, deps. Cheap — run before the model load to catch problems early.
import subprocess
import torch
import transformers

print('=== GPU ===')
try:
	print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], text=True))
except FileNotFoundError:
	print('nvidia-smi not found — CPU-only environment, model load will OOM')

print('=== Disk (cwd) ===')
print(subprocess.check_output(['df', '-h', '.'], text=True))

print('=== Deps ===')
print(f'torch={torch.__version__}, cuda={torch.version.cuda}, transformers={transformers.__version__}')
print(f'cuda available: {torch.cuda.is_available()}, devices: {torch.cuda.device_count()}')

# Llama-3.1-8B bf16 ≈ 16 GB weights; need ~20 GB free with activations + KV cache headroom
if torch.cuda.is_available():
	free_gb = torch.cuda.mem_get_info()[0] / 1e9
	print(f'free VRAM: {free_gb:.1f} GB')
	if free_gb < 20:
		print('warning: <20 GB free VRAM. Llama-8B bf16 may OOM during generation.')


## Cell 1.45 — confirm HF Write access to the artifact repo

Same fail-fast probe as M1/M2 Llama. M3 is the longest run (~6-8 GPU-hr) and the checkpoint-sync loop inside `run_eval` calls HF every 500 prompts. A 403 here would silently no-op all checkpoint uploads for the entire run — making session-death recovery impossible. Verify Write access NOW.

In [ ]:
# whose HF token is active, and does it have Write access to the artifact repo?
# fail fast — discovering the 403 after 8 GPU-hours is a budget-killer. This probe
# uploads a tiny placeholder then immediately deletes it, so nothing ends up visible
# in the repo's file tree. (Commits remain in history — that's how HF works.)
import os
import tempfile
from huggingface_hub import whoami, upload_file, delete_file

from src.lib.config import load_config
cfg = load_config()
REPO_ID = cfg['paths']['hf_artifact_repo']

who = whoami()
HF_USER = who['name']
# expose to later cells so HF upload commit messages can attribute the run
os.environ['HF_USER'] = HF_USER
print(f'HF identity: {HF_USER}')
print(f'target repo: {REPO_ID}')

# write-access probe: upload then delete
_probe_in_repo = '.write_access_probe'
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as _f:
	_f.write('write-access probe (auto-deleted)\n')
	_probe_path = _f.name
try:
	upload_file(
		path_or_fileobj=_probe_path,
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: write-access check',
	)
except Exception as e:
	raise RuntimeError(
		f'\nHF Write to {REPO_ID} FAILED for user {HF_USER}:\n  {e}\n\n'
		f'Fix: the repo owner needs to add you as a Write collaborator at\n'
		f'  https://huggingface.co/datasets/{REPO_ID}/settings\n'
		f'Owner navigates to Settings -> Collaborators -> Add user -> {HF_USER} -> Write.\n'
		f'Stop the notebook here; do NOT burn GPU-hours until this is fixed.'
	)

# clean up so the probe doesn't show in the file tree of a public repo
try:
	delete_file(
		path_in_repo=_probe_in_repo,
		repo_id=REPO_ID,
		repo_type='dataset',
		commit_message='probe: cleanup',
	)
	print(f'OK — Write access confirmed; probe file deleted from repo tree.')
except Exception as e:
	print(f'warning: probe upload OK but delete failed ({e}); the placeholder file may be visible at HEAD until manually removed')


## Cell 1.5 — pull existing M3 Llama checkpoints from HF

Colab sessions die; checkpoints live on HF. Rehydrates partial progress so the `run_eval` calls below resume cleanly. Safe on a fresh dataset — failures are non-fatal.

In [ ]:
import sys
sys.path.insert(0, '.')
from pathlib import Path
from src.lib.config import load_config, layer_suffix

cfg = load_config()
lsuf = layer_suffix(cfg, 'llama')
outputs_dir = Path(cfg['paths']['outputs_dir']) / 'm3' / lsuf
outputs_dir.mkdir(parents=True, exist_ok=True)

try:
	from huggingface_hub import snapshot_download
	snapshot_download(
		repo_id=cfg['paths']['hf_artifact_repo'],
		repo_type='dataset',
		allow_patterns=[f'm3_checkpoints/{lsuf}/*', f'm3/{lsuf}/*'],
		local_dir='outputs',
	)
	checkpoint_src = Path(f'outputs/m3_checkpoints/{lsuf}')
	if checkpoint_src.exists():
		for f in checkpoint_src.glob('*.csv'):
			dest = outputs_dir / f.name
			if not dest.exists():
				f.rename(dest)
	print(f'pulled checkpoints into {outputs_dir}:')
	for f in sorted(outputs_dir.glob('*.csv')):
		print(f' - {f.name}')
except Exception as e:
	print(f'no prior checkpoints found on HF (or pull failed): {e}')
	print('starting M3 Llama from scratch')


## Cell 2 — load model, desperation vector, calibrate ||h||

In [ ]:
from pathlib import Path
import numpy as np

from src.lib.model_load import load_llama
from src.steering import load_emotion_vector, estimate_residual_norm

# Toggle for smoke test vs full run. None = all 2,492 FaithEval prompts per arm.
LIMIT = None

layer = cfg['models']['llama']['extraction_layer']
alphas = cfg['steering']['alpha_sweep']

desperation = load_emotion_vector('desperation', model_key='llama')
print(f'desperation vector: shape={desperation.shape}, ||v||={np.linalg.norm(desperation):.4f}')

model, tokenizer = load_llama()
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, steering layer={layer}')

data_dir = Path(cfg['paths']['data_dir'])
neutral_texts = [p.read_text(encoding='utf-8') for p in sorted((data_dir / 'stories' / 'neutral').glob('*.txt'))]
norm_scale = estimate_residual_norm(
    model, tokenizer, layer=layer,
    calibration_texts=neutral_texts,
    token_skip=cfg['extraction']['token_skip'],
)
print(f'mean ||h||@L{layer} = {norm_scale:.2f}')
print(f'α-sweep: {alphas}')
print(f'LIMIT = {LIMIT} (None means all 2,492 prompts per arm)')


## Cell 3 — baseline arm (no hook)

~1 hr on A100 at full LIMIT=None for 8B. Checkpoint CSV saves every 500 prompts so a crash mid-arm resumes cleanly via the same path.

In [ ]:
from src.faitheval_eval import run_eval, summary

df_baseline = run_eval(
    model, tokenizer,
    limit=LIMIT,
    pre_forward_hook=None,
    checkpoint_path=outputs_dir / 'faitheval_baseline.csv',
    checkpoint_every=500,
    hf_sync_repo=cfg['paths']['hf_artifact_repo'],
    hf_sync_path=f'm3_checkpoints/{lsuf}/faitheval_baseline.csv',
)
baseline_summary = summary(df_baseline)
print(baseline_summary)


## Cell 4 — steered arm (dose-response, M2-validated α)

Four runs at α ∈ {0.025, 0.05, 0.075, 0.1}, each with the desperation steering hook. ~4 hr total at full LIMIT=None. Each α's CSV is separate so partial completion is durable.

In [ ]:
from src.steering import make_steering_hook_factory

steered_summaries = {}
for alpha in alphas:
    print(f'\n=== steered α={alpha} ===')
    hook_factory = make_steering_hook_factory(
        vector=desperation, layer=layer, alpha=alpha, norm_scale=norm_scale,
    )
    df = run_eval(
        model, tokenizer,
        limit=LIMIT,
        pre_forward_hook=hook_factory,
        checkpoint_path=outputs_dir / f'faitheval_steered_a{alpha}.csv',
        checkpoint_every=500,
        hf_sync_repo=cfg['paths']['hf_artifact_repo'],
        hf_sync_path=f'm3_checkpoints/{lsuf}/faitheval_steered_a{alpha}.csv',
    )
    s = summary(df)
    steered_summaries[alpha] = s
    print(s)


## Cell 5 — ablated arm (project-out)

Orthogonally project the desperation direction OUT of the residual stream at the steering layer on every forward pass. No α — hard projection. ~1 hr at full LIMIT=None.

In [ ]:
from src.steering import make_ablation_hook_factory

print('=== ablated (project-out desperation) ===')
ablation_factory = make_ablation_hook_factory(vector=desperation, layer=layer)
df_ablated = run_eval(
    model, tokenizer,
    limit=LIMIT,
    pre_forward_hook=ablation_factory,
    checkpoint_path=outputs_dir / 'faitheval_ablated.csv',
    checkpoint_every=500,
    hf_sync_repo=cfg['paths']['hf_artifact_repo'],
    hf_sync_path=f'm3_checkpoints/{lsuf}/faitheval_ablated.csv',
)
ablated_summary = summary(df_ablated)
print(ablated_summary)


## Cell 6 — aggregate, dose-response, decision

Build the dose-response table, check monotonicity, check the baseline-vs-ablated gap, write the decision file.

In [ ]:
import pandas as pd

rows = []
rows.append({'arm': 'baseline', 'alpha': 0.0, **baseline_summary})
for a in alphas:
    rows.append({'arm': 'steered', 'alpha': a, **steered_summaries[a]})
rows.append({'arm': 'ablated', 'alpha': None, **ablated_summary})

dose = pd.DataFrame(rows)
dose.to_csv(outputs_dir / 'dose_response.csv', index=False)
print(dose.to_string(index=False))

steered = dose[dose['arm'] == 'steered'].sort_values('alpha')
halluc_series = steered['hallucination_rate'].tolist()
monotonic = all(halluc_series[i] <= halluc_series[i+1] for i in range(len(halluc_series)-1))

baseline_h = baseline_summary['hallucination_rate']
ablated_h = ablated_summary['hallucination_rate']
ablation_gap = baseline_h - ablated_h

lines = ['Module 3 — FaithEval dose-response + ablation (Llama-3.1-8B-Instruct)', '=' * 60, '']
lines.append(f'n_prompts per arm: {baseline_summary["n"]}')
lines.append(f'norm scale ||h||@L{layer}: {norm_scale:.2f}')
lines.append('')
lines.append('hallucination rates:')
lines.append(f'  baseline  (α=0)    : {baseline_h:.4f}')
for a in alphas:
    lines.append(f'  steered   α={a:.3f}: {steered_summaries[a]["hallucination_rate"]:.4f}')
lines.append(f'  ablated   (proj-out): {ablated_h:.4f}')
lines.append('')
lines.append(f'monotonic dose-response: {monotonic}')
lines.append(f'ablation gap (baseline − ablated): {ablation_gap:+.4f} (positive = ablation reduced hallucination)')

decision = '\n'.join(lines)
print('\n' + decision)
(outputs_dir / 'decision.txt').write_text(decision, encoding='utf-8')


## Push artifacts to HF Hub

In [ ]:
import os
from huggingface_hub import HfApi, create_repo

repo_id = cfg['paths']['hf_artifact_repo']
create_repo(repo_id, repo_type='dataset', private=True, exist_ok=True)
hf_user = os.environ.get('HF_USER', 'unknown')

info = HfApi().upload_folder(
        folder_path=f'outputs/m3/{lsuf}',
        repo_id=repo_id,
        repo_type='dataset',
        path_in_repo=f'm3_results/{lsuf}',
        commit_message=f'M3 Llama-3.1-8B-Instruct: baseline + steered (M2 α) + ablated (by HF user {hf_user})',
)
print('uploaded ->', info.commit_url if hasattr(info, 'commit_url') else info)


## Outputs

- `outputs/m3/llama_L{layer}/faitheval_baseline.csv` — per-prompt no-hook results
- `outputs/m3/llama_L{layer}/faitheval_steered_a{α}.csv` — per α steered results
- `outputs/m3/llama_L{layer}/faitheval_ablated.csv` — project-out arm
- `outputs/m3/llama_L{layer}/dose_response.csv` — aggregated rate-per-arm table
- `outputs/m3/llama_L{layer}/decision.txt` — monotonicity + ablation-gap summary

## Interpretation (human-owned)

Same caveats as M3 Gemma: monotonicity on noisy 2,492-prompt rates is not a meaningful dose-response on its own (bootstrap CIs before claiming an effect); watch the refuse/fabricate/off_topic decomposition; the FaithEval prompt template's explicit 'unknown' hint inflates baseline refusal. If results look interesting, the Gemma notebook's diagnostic cells (`fabricate_coherence_audit`, `leakage_scan`, `retention_table`) are reasonable next-step ports.